In [7]:
import json
import pickle
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display, Markdown


def find_project_root(start=None):
    path = Path.cwd() if start is None else Path(start)
    for candidate in [path, *path.parents]:
        if (candidate / "pipelines").exists() and (candidate / "spectral_code").exists():
            return candidate
    raise RuntimeError("Could not find Spectral-Software project root.")

PROJECT_ROOT = find_project_root()
BCB_DUMP = Path(r"C:\Users\koush\PyProjects\bcb")
OUTPUT_BASE = PROJECT_ROOT.parent / "outputs"
ANALYSIS_OUT = PROJECT_ROOT / "notebooks" / "analysis" / "outputs"
ANALYSIS_OUT.mkdir(parents=True, exist_ok=True)

DATASET_TYPES = [1, 2, 3, 4]
GRAPH_TYPES = ["ast", "cfg", "ddg", "pdg", "cpg"]


def type_paths(dataset_type):
    return {
        "bench_dir": PROJECT_ROOT / "bench_data" / f"bcb_full_type{dataset_type}",
        "output_root": OUTPUT_BASE / f"type{dataset_type}",
        "spectral_manifest": OUTPUT_BASE / f"type{dataset_type}" / "spectral_features" / "spectral_features_manifest.json",
    }


def available_dataset_types():
    available = []
    for dataset_type in DATASET_TYPES:
        paths = type_paths(dataset_type)
        if (paths["bench_dir"] / "train.txt").exists() and paths["spectral_manifest"].exists():
            available.append(dataset_type)
        else:
            print(f"Skipping Type {dataset_type}: missing train.txt or spectral manifest.")
    return available

AVAILABLE_TYPES = available_dataset_types()
print("Available dataset types:", AVAILABLE_TYPES)


def normalize_pair(left, right):
    left = int(left)
    right = int(right)
    return (left, right) if left <= right else (right, left)


def load_pairs(path):
    rows = []
    with Path(path).open("r", encoding="utf-8") as f:
        for line in f:
            left, right, label = line.strip().split("\t")
            rows.append((int(left), int(right), int(label)))
    return pd.DataFrame(rows, columns=["left_id", "right_id", "label"])


def iter_copy_table(dump_path, table_name, desc):
    dump_path = Path(dump_path)
    target_prefix = f"COPY {table_name} "
    in_table = False
    with dump_path.open("rb") as f, tqdm(total=dump_path.stat().st_size, unit="B", unit_scale=True, desc=desc) as bar:
        for raw in f:
            bar.update(len(raw))
            line = raw.decode("utf-8", errors="replace").rstrip("\n")
            if not in_table:
                if line.startswith(target_prefix):
                    in_table = True
                continue
            if line == r"\.":
                break
            yield line


def parse_float(raw):
    return None if raw == r"\N" or raw == "" else float(raw)


def copy_unescape(value):
    if value == r"\N":
        return ""
    result = []
    i = 0
    while i < len(value):
        char = value[i]
        if char != "\\" or i + 1 >= len(value):
            result.append(char)
            i += 1
            continue
        escaped = value[i + 1]
        replacements = {"n": "\n", "r": "\r", "t": "\t", "\\": "\\"}
        result.append(replacements.get(escaped, escaped))
        i += 2
    return "".join(result)


def load_functionality_lookup(cache_path=None):
    if cache_path and Path(cache_path).exists():
        return pd.read_csv(cache_path)

    rows = []
    for line in iter_copy_table(BCB_DUMP, "public.functionalities", "Scanning BCB functionality lookup"):
        parts = line.split("\t")
        if len(parts) < 4:
            continue
        rows.append({
            "functionality_name": copy_unescape(parts[0]),
            "functionality_description": copy_unescape(parts[1]),
            "functionality_id": int(parts[2]),
            "search_heuristic": copy_unescape(parts[3]),
        })

    lookup = pd.DataFrame(rows)
    if cache_path:
        lookup.to_csv(cache_path, index=False)
    return lookup


def load_clone_metadata_for_pair_keys(pair_keys, cache_path=None):
    pair_keys = set(pair_keys)
    if cache_path and Path(cache_path).exists():
        cached = pd.read_csv(cache_path)
        cached_keys = set(zip(cached.left_id.astype(int), cached.right_id.astype(int)))
        if pair_keys.issubset(cached_keys):
            return cached

    rows = []
    found = set()
    for line in iter_copy_table(BCB_DUMP, "public.clones", "Scanning BCB clone metadata"):
        parts = line.split("\t")
        if len(parts) < 7:
            continue
        key = normalize_pair(parts[0], parts[1])
        if key not in pair_keys or key in found:
            continue
        sim_line = parse_float(parts[5])
        sim_token = parse_float(parts[6])
        rows.append({
            "left_id": key[0],
            "right_id": key[1],
            "functionality_id": int(parts[2]),
            "bcb_type": parts[3],
            "syntactic_type": int(parts[4]),
            "similarity_line": sim_line,
            "similarity_token": sim_token,
            "min_similarity": min(sim_line, sim_token),
        })
        found.add(key)
        if len(found) == len(pair_keys):
            break

    missing = pair_keys - found
    if missing:
        print(f"Warning: {len(missing):,} pair keys were not found in public.clones.")
    meta = pd.DataFrame(rows)
    if cache_path:
        meta.to_csv(cache_path, index=False)
    return meta


def load_spectral_features_for_ids(dataset_type, method_ids):
    method_ids = {str(int(mid)) for mid in method_ids}
    manifest_path = type_paths(dataset_type)["spectral_manifest"]
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    features = {}
    for shard_path in tqdm(manifest["shards"], desc=f"Type {dataset_type}: spectral shards", unit="shard"):
        with open(shard_path, "rb") as f:
            shard = pickle.load(f)
        missing = method_ids - set(features)
        for method_id in list(missing):
            if method_id in shard:
                features[method_id] = shard[method_id]
        if len(features) == len(method_ids):
            break
    return features


def informative_eigenvalues(record, graph_type, eps=1e-10):
    if not record:
        return None
    values = np.asarray(record.get(graph_type, {}).get("eigenvalues", []), dtype=float)
    if values.size == 0 or not np.all(np.isfinite(values)) or not np.any(np.abs(values) > eps):
        return None
    return values


def node_count(record, graph_type):
    if not record:
        return np.nan
    return record.get(graph_type, {}).get("nodes", np.nan)


def pss(ev1, ev2):
    ev1 = np.asarray(ev1, dtype=float)
    ev2 = np.asarray(ev2, dtype=float)
    nz1 = np.nonzero(ev1)[0]
    nz2 = np.nonzero(ev2)[0]
    if len(nz1) == 0 or len(nz2) == 0:
        return np.nan
    ev1 = ev1[:nz1[-1] + 1]
    ev2 = ev2[:nz2[-1] + 1]
    max_len = max(len(ev1), len(ev2))
    if len(ev1) != max_len:
        ev1 = np.interp(np.linspace(0, 1, max_len), np.linspace(0, 1, len(ev1)), ev1)
    if len(ev2) != max_len:
        ev2 = np.interp(np.linspace(0, 1, max_len), np.linspace(0, 1, len(ev2)), ev2)
    n1 = np.linalg.norm(ev1)
    n2 = np.linalg.norm(ev2)
    if n1 == 0 or n2 == 0:
        return np.nan
    distance = np.linalg.norm(ev1 / n1 - ev2 / n2)
    return float(np.clip((np.sqrt(2) - distance) / np.sqrt(2), 0.0, 1.0))

Available dataset types: [1, 2, 3, 4]


In [8]:
positive_frames = []
for dataset_type in AVAILABLE_TYPES:
    bench_dir = type_paths(dataset_type)["bench_dir"]
    path = bench_dir / "train_positives.txt"
    df = load_pairs(path)
    df["dataset_type"] = dataset_type
    df["pair_key"] = [normalize_pair(l, r) for l, r in zip(df.left_id, df.right_id)]
    positive_frames.append(df)

positive_pairs = pd.concat(positive_frames, ignore_index=True)
display(positive_pairs.groupby("dataset_type").size().rename("positive_pairs").reset_index())

,dataset_type,positive_pairs
0,1,48062
1,2,4230
2,3,30000
3,4,30000


In [9]:
metadata_cache = ANALYSIS_OUT / "all_types_positive_clone_metadata.csv"
functionality_cache = ANALYSIS_OUT / "bcb_functionalities.csv"
all_pair_keys = set(positive_pairs["pair_key"])
clone_meta = load_clone_metadata_for_pair_keys(all_pair_keys, cache_path=metadata_cache)
clone_meta["pair_key"] = [normalize_pair(l, r) for l, r in zip(clone_meta.left_id, clone_meta.right_id)]
functionality_lookup = load_functionality_lookup(cache_path=functionality_cache)

positive_meta = positive_pairs.merge(
    clone_meta.drop(columns=["left_id", "right_id"]),
    on="pair_key",
    how="left",
).merge(
    functionality_lookup,
    on="functionality_id",
    how="left",
)
positive_meta.to_csv(ANALYSIS_OUT / "all_types_positive_clone_metadata_with_functionality.csv", index=False)

display(positive_meta.head())
display(
    positive_meta
    .groupby(["dataset_type", "functionality_id", "functionality_name"])
    .size()
    .rename("pairs")
    .reset_index()
    .sort_values(["dataset_type", "pairs"], ascending=[True, False])
    .head(30)
)


Scanning BCB functionality lookup:   0%|          | 0.00/14.0G [00:00<?, ?B/s]

,left_id,right_id,label,dataset_type,pair_key,functionality_id,bcb_type,syntactic_type,similarity_line,similarity_token,min_similarity,functionality_name,functionality_description,search_heuristic
0,74,789253,1,1,"(74, 789253)",4,tagged-tagged,1,1.0,1.0,1.0,Copy File,Copies a file.,[getChannel] OR [transferFrom] OR [FileUtils.c...
1,661,7667,1,1,"(661, 7667)",4,tagged-tagged,1,1.0,1.0,1.0,Copy File,Copies a file.,[getChannel] OR [transferFrom] OR [FileUtils.c...
2,661,9563,1,1,"(661, 9563)",4,tagged-tagged,1,1.0,1.0,1.0,Copy File,Copies a file.,[getChannel] OR [transferFrom] OR [FileUtils.c...
3,661,38361,1,1,"(661, 38361)",4,tagged-tagged,1,1.0,1.0,1.0,Copy File,Copies a file.,[getChannel] OR [transferFrom] OR [FileUtils.c...
4,661,44823,1,1,"(661, 44823)",4,tagged-tagged,1,1.0,1.0,1.0,Copy File,Copies a file.,[getChannel] OR [transferFrom] OR [FileUtils.c...


,dataset_type,functionality_id,functionality_name,pairs
2,1,4,Copy File,13802
12,1,20,Fibonacci,11194
35,1,44,Test Palindrome,10879
19,1,27,Call Method Using Reflection,3552
23,1,31,File Dialog,1892
0,1,2,Download From Web,1553
26,1,34,Execute External Process,1484
1,1,3,Secure Hash,632
31,1,40,Parse CSV File,482
24,1,32,Send E-Mail,453


In [10]:
rows = []
for dataset_type, group in positive_meta.groupby("dataset_type"):
    needed_ids = set(group.left_id) | set(group.right_id)
    features = load_spectral_features_for_ids(dataset_type, needed_ids)
    for pair in tqdm(group.itertuples(index=False), total=len(group), desc=f"Type {dataset_type}: positive pair PSS"):
        left_record = features.get(str(pair.left_id))
        right_record = features.get(str(pair.right_id))
        for graph_type in GRAPH_TYPES:
            left_ev = informative_eigenvalues(left_record, graph_type)
            right_ev = informative_eigenvalues(right_record, graph_type)
            if left_ev is None or right_ev is None:
                score = np.nan
            else:
                score = pss(left_ev, right_ev)
            rows.append({
                "dataset_type": dataset_type,
                "left_id": pair.left_id,
                "right_id": pair.right_id,
                "functionality_id": pair.functionality_id,
                "functionality_name": pair.functionality_name,
                "syntactic_type": pair.syntactic_type,
                "min_similarity": pair.min_similarity,
                "graph_type": graph_type,
                "pss": score,
            })

pair_pss = pd.DataFrame(rows)
pair_pss.to_csv(ANALYSIS_OUT / "all_types_functionality_pair_pss.csv", index=False)
display(pair_pss.head())

Type 1: spectral shards:   0%|          | 0/108 [00:00<?, ?shard/s]

Type 1: positive pair PSS:   0%|          | 0/48062 [00:00<?, ?it/s]

Type 2: spectral shards:   0%|          | 0/192 [00:00<?, ?shard/s]

Type 2: positive pair PSS:   0%|          | 0/4230 [00:00<?, ?it/s]

Type 3: spectral shards:   0%|          | 0/147 [00:00<?, ?shard/s]

Type 3: positive pair PSS:   0%|          | 0/30000 [00:00<?, ?it/s]

Type 4: spectral shards:   0%|          | 0/152 [00:00<?, ?shard/s]

Type 4: positive pair PSS:   0%|          | 0/30000 [00:00<?, ?it/s]

,dataset_type,left_id,right_id,functionality_id,functionality_name,syntactic_type,min_similarity,graph_type,pss
0,1,74,789253,4,Copy File,1,1.0,ast,1.0
1,1,74,789253,4,Copy File,1,1.0,cfg,1.0
2,1,74,789253,4,Copy File,1,1.0,ddg,1.0
3,1,74,789253,4,Copy File,1,1.0,pdg,1.0
4,1,74,789253,4,Copy File,1,1.0,cpg,1.0


In [11]:
summary = (
    pair_pss.dropna(subset=["pss"])
    .groupby(["dataset_type", "functionality_id", "functionality_name", "graph_type"])
    .agg(
        pairs=("pss", "size"),
        pss_mean=("pss", "mean"),
        pss_median=("pss", "median"),
        pss_std=("pss", "std"),
        min_similarity_mean=("min_similarity", "mean"),
    )
    .reset_index()
    .sort_values(["dataset_type", "graph_type", "pairs"], ascending=[True, True, False])
)
summary.to_csv(ANALYSIS_OUT / "all_types_functionality_pss_summary.csv", index=False)
display(summary.head(50))


,dataset_type,functionality_id,functionality_name,graph_type,pairs,pss_mean,pss_median,pss_std,min_similarity_mean
60,1,20,Fibonacci,ast,11193,1.0,1.0,8.210461e-17,1.0
175,1,44,Test Palindrome,ast,10878,1.0,1.0,4.450680e-17,1.0
10,1,4,Copy File,ast,8358,1.0,1.0,1.123232e-16,1.0
95,1,27,Call Method Using Reflection,ast,2946,1.0,1.0,1.191153e-16,1.0
115,1,31,File Dialog,ast,1878,1.0,1.0,6.365326e-17,1.0
0,1,2,Download From Web,ast,1517,1.0,1.0,5.738365e-17,1.0
130,1,34,Execute External Process,ast,1436,1.0,1.0,1.146009e-16,1.0
5,1,3,Secure Hash,ast,494,1.0,1.0,1.113596e-16,1.0
160,1,41,Transpose a Matrix.,ast,431,1.0,1.0,5.050926e-17,1.0
150,1,39,Delete Folder and Contents,ast,250,1.0,1.0,6.213808e-17,1.0


In [12]:
# Compact pivot for comparing graph layers inside each functionality group.
pivot = summary.pivot_table(
    index=["dataset_type", "functionality_id", "functionality_name"],
    columns="graph_type",
    values="pss_mean",
    aggfunc="mean",
).reset_index()
pair_counts = (
    positive_meta
    .groupby(["dataset_type", "functionality_id", "functionality_name"])
    .size()
    .rename("positive_pairs")
    .reset_index()
)
pivot = pivot.merge(pair_counts, on=["dataset_type", "functionality_id", "functionality_name"], how="left")
pivot = pivot.sort_values(["dataset_type", "positive_pairs"], ascending=[True, False])
pivot.to_csv(ANALYSIS_OUT / "all_types_functionality_pss_pivot.csv", index=False)
display(pivot.head(50))


,dataset_type,functionality_id,functionality_name,ast,cfg,cpg,ddg,pdg,positive_pairs
2,1,4,Copy File,1.0,1.0,0.999998,0.999993,0.999991,13802
12,1,20,Fibonacci,1.0,1.0,1.000000,1.000000,1.000000,11194
35,1,44,Test Palindrome,1.0,1.0,1.000000,1.000000,1.000000,10879
19,1,27,Call Method Using Reflection,1.0,1.0,1.000000,1.000000,1.000000,3552
23,1,31,File Dialog,1.0,1.0,1.000000,1.000000,1.000000,1892
0,1,2,Download From Web,1.0,1.0,1.000000,1.000000,1.000000,1553
26,1,34,Execute External Process,1.0,1.0,1.000000,1.000000,1.000000,1484
1,1,3,Secure Hash,1.0,1.0,1.000000,1.000000,1.000000,632
31,1,40,Parse CSV File,1.0,1.0,1.000000,1.000000,1.000000,482
24,1,32,Send E-Mail,1.0,1.0,1.000000,1.000000,1.000000,453
